# Spatial Cross-Validation via Spherical K-MeansThis notebook implements **cluster-based spatial partitioning** for species distribution models, producing spatially blocked CV folds and a held-out external test set.## Strategy1. **Spherical K-Means** on presence points (3-D unit vectors on the globe)  2. Background points labelled by nearest cluster centroid  3. A subset of clusters is reserved as a **fixed external test set**  4. Remaining clusters are assigned to **K CV folds** via greedy bin-packing (balanced by presence and background counts)  5. An optional **geographic buffer** flags CV points that are close to the test set  ```Presences ──► Spherical K-Means (N clusters)                     │              ┌──────┴──────┐         Test clusters    CV clusters              │              │              │         Greedy fold              │         assignment              │              │              ▼              ▼         test_split.csv   cv_split.csv```

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.cluster import KMeans
from pathlib import Path
import warnings
import matplotlib.pyplot as plt

## 2. Configuration

In [ ]:
# ── Clustering ────────────────────────────────────────────
N_CLUSTERS      = 50       # total spherical clusters
N_TEST_CLUSTERS = 5        # clusters reserved for external test
N_FOLDS         = 10       # CV folds from remaining clusters
SEED            = 42

# ── Test cluster selection ────────────────────────────────
# Options: "staggered", "top", "random_presence", "mixed_margins"
TEST_SELECTION_STRATEGY = "staggered"

# ── Geographic buffer ─────────────────────────────────────
TEST_BUFFER_KM = 300.0     # flag CV points within this distance of test presences (km)

# ── Fold balance constraints ──────────────────────────────
MIN_BG_PER_FOLD = 300

# ── Input data ────────────────────────────────────────────
# Option A: load from a GeoPackage (presence + background layers)
INPUT_GPKG         = "data/points/tgb_outputs.gpkg"
LAYER_PRESENCES    = "presences"
LAYER_BACKGROUND   = "background"

# Option B: load presences from CSV (uncomment and set paths)
# DENGUE_CSV = "data/points/dengue_presences.csv"
# LON_COL, LAT_COL = "Longitude", "Latitude"

# ── Output ────────────────────────────────────────────────
OUT_GPKG               = "data/points/tgb_outputs.gpkg"
OUT_LAYER_PRES_SPLITS  = "presences_sphericalcv_splits"
OUT_LAYER_BG_SPLITS    = "background_sphericalcv_splits"

TEST_CSV = "data/splits/test_split.csv"
CV_CSV   = "data/splits/cv_split.csv"

# ── Diagnostics ───────────────────────────────────────────
RUN_DIAGNOSTICS = True
RUN_PLOTS       = False

## 3. Spherical Geometry HelpersAll distance calculations use great-circle geometry on the WGS84 sphere (R ≈ 6371 km).

In [ ]:
R_KM = 6371.0088  # mean Earth radius


def _assert_crs_wgs84(gdf, name="GeoDataFrame"):
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        raise ValueError(f"{name} must be WGS84 (EPSG:4326). Got: {gdf.crs}")


def spherical_unit_vectors(gdf):
    """Convert lon/lat points to 3-D unit vectors on the sphere."""
    lon = np.deg2rad(gdf.geometry.x.values)
    lat = np.deg2rad(gdf.geometry.y.values)
    return np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


def gc_distance_km(U1, U2):
    """Great-circle distance matrix (km) between two sets of unit vectors."""
    dots = np.clip(U1 @ U2.T, -1.0, 1.0)
    return R_KM * np.arccos(dots)


def min_gc_distance_to_set_km(U_points, U_set):
    """Minimum great-circle distance (km) from each point to the nearest point in a set."""
    dots = np.clip(U_points @ U_set.T, -1.0, 1.0)
    return R_KM * np.min(np.arccos(dots), axis=1)

## 4. K-Means ClusteringFit spherical K-Means on **presence points only**, then assign background points to the nearest cluster centroid.

In [ ]:
def fit_presence_kmeans(dengue_wgs, bg_wgs, n_clusters, seed):
    """
    Fit K-Means on presence unit vectors; label background by predict.
    Returns (pres_labels, bg_labels, kmeans_model).
    """
    X_pres = spherical_unit_vectors(dengue_wgs)
    X_bg   = spherical_unit_vectors(bg_wgs)

    km = KMeans(n_clusters=n_clusters, n_init=20, random_state=seed)
    pres_labels = km.fit_predict(X_pres)
    bg_labels   = km.predict(X_bg)
    return pres_labels, bg_labels, km


def _summarize_clusters(labels_pres, labels_bg):
    """Per-cluster presence and background counts."""
    pres = pd.Series(labels_pres).value_counts().rename("n_pres")
    bg   = pd.Series(labels_bg).value_counts().rename("n_bg")
    total = pres.add(bg, fill_value=0).rename("n_all")
    df = pd.concat([pres, bg, total], axis=1).fillna(0).astype(int).reset_index()
    return df.rename(columns={"index": "cluster"})

## 5. Test Cluster SelectionChoose which clusters become the fixed external test set.| Strategy | Description ||----------|-------------|| `staggered` | Evenly spaced indices through presence-sorted clusters (default) || `top` | Clusters with the most presences || `random_presence` | Random subset of presence-bearing clusters || `mixed_margins` | Half from densest, half from sparsest presence clusters |

In [ ]:
def choose_test_clusters(cluster_stats, n_test, rng, min_pres=1,
                        strategy=TEST_SELECTION_STRATEGY):
    """Select cluster IDs for the external test set."""
    if n_test <= 0:
        return np.array([], dtype=int)

    total = len(cluster_stats)
    if n_test >= total:
        warnings.warn("n_test >= total clusters; capping at 20%.")
        n_test = max(1, total // 5)

    cands = cluster_stats[cluster_stats["n_pres"] >= min_pres].copy()
    if cands.empty:
        raise RuntimeError("No clusters with presences for test selection.")

    cands = cands.sort_values(["n_pres", "n_all"], ascending=[False, False]).reset_index(drop=True)

    if strategy == "top":
        chosen = cands.iloc[:n_test]

    elif strategy == "random_presence":
        idx = rng.choice(len(cands), size=min(n_test, len(cands)), replace=False)
        chosen = cands.iloc[idx]

    elif strategy == "mixed_margins":
        n_top = n_test // 2
        n_margin = n_test - n_top
        top = cands.iloc[:min(n_top, len(cands))]
        marginals = cands.sort_values(["n_pres", "n_all"], ascending=True).reset_index(drop=True)
        if len(marginals) > 0 and n_margin > 0:
            idx = rng.choice(len(marginals), size=min(n_margin, len(marginals)), replace=False)
            marginals = marginals.iloc[idx]
        else:
            marginals = pd.DataFrame(columns=cands.columns)
        chosen = pd.concat([top, marginals]).drop_duplicates().iloc[:n_test]

    else:  # staggered
        step = max(1, len(cands) // n_test)
        start = int(rng.integers(0, min(step, len(cands))))
        idxs = list(range(start, min(start + step * n_test, len(cands)), step))
        if len(idxs) < n_test:
            remaining = list(set(range(len(cands))) - set(idxs))
            rng.shuffle(remaining)
            idxs += remaining[:n_test - len(idxs)]
        chosen = cands.iloc[idxs]

    return chosen["cluster"].to_numpy(dtype=int)

## 6. Fold AssignmentGreedy bin-packing assigns CV clusters to folds, prioritising:1. Every fold gets at least one presence-bearing cluster  2. Background count per fold ≥ `MIN_BG_PER_FOLD` (soft constraint)  3. Overall balance across folds

In [ ]:
def balanced_fold_assignment(cluster_stats_cv, n_folds, rng,
                            min_pres_per_fold=1, min_bg_per_fold=MIN_BG_PER_FOLD):
    """Assign clusters to folds via greedy bin-packing.  Returns {cluster_id: fold_number}."""
    folds = [{"n_pres": 0, "n_bg": 0, "n_all": 0, "clusters": []} for _ in range(n_folds)]

    sorted_df = cluster_stats_cv.sort_values(
        ["n_pres", "n_bg", "n_all"], ascending=[False, False, False]
    )
    with_pres = sorted_df[sorted_df["n_pres"] > 0].to_dict("records")
    bg_only   = sorted_df[sorted_df["n_pres"] == 0].to_dict("records")

    def _pick_fold(c, enforce_bg=False):
        """Choose the best fold for cluster c."""
        if c["n_pres"] > 0:
            # prioritise folds still missing presences
            empty = [i for i, f in enumerate(folds) if f["n_pres"] < min_pres_per_fold]
            if empty:
                return min(empty, key=lambda i: folds[i]["n_all"])
        elif enforce_bg:
            under = [i for i, f in enumerate(folds) if f["n_bg"] < min_bg_per_fold]
            if under:
                return min(under, key=lambda i: folds[i]["n_bg"])

        # default: least-loaded fold
        return int(np.lexsort((
            [f["n_all"] for f in folds],
            [f["n_bg"]  for f in folds],
            [f["n_pres"] for f in folds],
        ))[0])

    for c in with_pres:
        i = _pick_fold(c)
        folds[i]["n_pres"] += c["n_pres"]
        folds[i]["n_bg"]   += c["n_bg"]
        folds[i]["n_all"]  += c["n_all"]
        folds[i]["clusters"].append(c["cluster"])

    for c in bg_only:
        i = _pick_fold(c, enforce_bg=True)
        folds[i]["n_pres"] += c["n_pres"]
        folds[i]["n_bg"]   += c["n_bg"]
        folds[i]["n_all"]  += c["n_all"]
        folds[i]["clusters"].append(c["cluster"])

    mapping = {}
    for k, f in enumerate(folds, start=1):
        for cl in f["clusters"]:
            mapping[cl] = k

    if any(f["n_pres"] < min_pres_per_fold for f in folds):
        raise RuntimeError("Fold assignment failed: a fold has too few presences.")
    if any(f["n_bg"] == 0 for f in folds):
        warnings.warn("Some folds have zero background points.")

    return mapping

## 7. Geographic Buffer

In [ ]:
def compute_test_buffer_mask(gdf, split_col, buffer_km):
    """
    Flag all points within `buffer_km` of any TEST presence.
    Returns a boolean array (True = inside buffer).
    """
    if buffer_km is None or buffer_km <= 0:
        return np.zeros(len(gdf), dtype=bool)

    test_mask = (gdf[split_col] == "test")
    if not test_mask.any():
        return np.zeros(len(gdf), dtype=bool)

    U_all  = spherical_unit_vectors(gdf)
    U_test = spherical_unit_vectors(gdf.loc[test_mask])

    return min_gc_distance_to_set_km(U_all, U_test) <= buffer_km

## 8. Diagnostics

In [ ]:
def leakage_diagnostics(pres_split):
    """Report distances between CV and test presences."""
    test_pts = pres_split.loc[pres_split["split"] == "test"]
    cv_pts   = pres_split.loc[pres_split["split"] == "cv"]

    if len(test_pts) == 0 or len(cv_pts) == 0:
        print("[Leakage] Not enough points for diagnostics.")
        return

    U_test = spherical_unit_vectors(test_pts)
    U_cv   = spherical_unit_vectors(cv_pts)
    d = min_gc_distance_to_set_km(U_cv, U_test)

    print("\n[Leakage: CV presences → nearest TEST presence]")
    print(f"  min:  {np.min(d):.1f} km")
    print(f"  p5:   {np.percentile(d, 5):.1f} km")
    print(f"  p25:  {np.percentile(d, 25):.1f} km")
    print(f"  med:  {np.median(d):.1f} km")
    print(f"  <100: {(d < 100).mean():.3f}")
    print(f"  <300: {(d < 300).mean():.3f}")


def fold_separation_diagnostics(pres_split):
    """Report inter-fold distances for CV presences."""
    cv = pres_split[pres_split["split"] == "cv"]
    folds = sorted(cv["fold"].dropna().unique())
    U_all = spherical_unit_vectors(pres_split)

    for f in folds:
        mask_f = (pres_split["split"] == "cv") & (pres_split["fold"] == f)
        mask_o = (pres_split["split"] == "cv") & (pres_split["fold"] != f)
        U_f, U_o = U_all[mask_f.values], U_all[mask_o.values]
        if len(U_f) == 0 or len(U_o) == 0:
            continue
        d = min_gc_distance_to_set_km(U_f, U_o)
        print(f"\n[Fold {int(f)}] min={np.min(d):.1f} km, med={np.median(d):.1f} km, "
              f"<100km: {(d < 100).mean():.3f}")

## 9. Plotting (Optional)

In [ ]:
def plot_test_and_buffer(pres_split):
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
    fig, ax = plt.subplots(figsize=(12, 6))
    world.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.3)

    pres_split.query("split == 'cv' and not in_test_buffer").plot(
        ax=ax, markersize=5, alpha=0.4, color="blue", label="CV (outside buffer)")
    pres_split.query("split == 'cv' and in_test_buffer").plot(
        ax=ax, markersize=5, alpha=0.6, color="orange", label="CV (in buffer)")
    pres_split.query("split == 'test'").plot(
        ax=ax, markersize=8, alpha=0.8, color="red", label="Test")

    ax.legend(); ax.set_title("Presences: test / buffer / CV")
    plt.tight_layout(); plt.show()


def plot_fold_counts(pres_split, bg_split):
    pres_c = pres_split.query("split == 'cv'").groupby("fold").size()
    bg_c   = bg_split.query("split == 'cv'").groupby("fold").size()
    folds = sorted(pres_c.index.astype(int))
    x = np.arange(len(folds))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.15, pres_c[folds], width=0.3, label="Presence")
    ax.bar(x + 0.15, bg_c.reindex(folds).fillna(0).values, width=0.3, label="Background")
    ax.set_xticks(x); ax.set_xticklabels(folds)
    ax.set_xlabel("Fold"); ax.set_ylabel("Count"); ax.legend()
    ax.set_title("Per-fold counts (CV)"); plt.tight_layout(); plt.show()


def plot_cv_to_test_distances(pres_split, bins=30):
    test = pres_split[pres_split["split"] == "test"]
    cv   = pres_split[pres_split["split"] == "cv"]
    if len(test) == 0 or len(cv) == 0:
        return
    d = min_gc_distance_to_set_km(spherical_unit_vectors(cv), spherical_unit_vectors(test))
    plt.figure(figsize=(6, 4))
    plt.hist(d, bins=bins)
    plt.xlabel("Distance to nearest TEST presence (km)")
    plt.ylabel("Frequency"); plt.title("CV → TEST distances")
    plt.tight_layout(); plt.show()

## 10. CSV Export

In [ ]:
def export_test_cv_csv(pres_split, bg_split, test_csv=TEST_CSV, cv_csv=CV_CSV):
    """
    Merge presence + background splits and export to CSV.

    Output columns: Longitude, Latitude, label, cluster_id, fold
    """
    pres = pres_split.copy(); pres["label"] = "presence"
    bg   = bg_split.copy();   bg["label"]   = "background"

    all_gdf = pd.concat([pres, bg], ignore_index=True)
    all_gdf["Longitude"] = all_gdf.geometry.x
    all_gdf["Latitude"]  = all_gdf.geometry.y

    cols = ["Longitude", "Latitude", "label", "cluster_id", "fold"]

    test_df = all_gdf[all_gdf["split"] == "test"][cols]
    cv_df   = all_gdf[all_gdf["split"] == "cv"][cols]

    Path(test_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(cv_csv).parent.mkdir(parents=True, exist_ok=True)

    test_df.to_csv(test_csv, index=False)
    cv_df.to_csv(cv_csv, index=False)

    print(f"Test: {test_csv}  ({len(test_df)} rows)")
    print(f"CV:   {cv_csv}  ({len(cv_df)} rows)")

---## 11. Run: Full Partitioning Pipeline

In [ ]:
rng = np.random.default_rng(SEED)

# ── Load data ─────────────────────────────────────────────
dengue_wgs = gpd.read_file(INPUT_GPKG, layer=LAYER_PRESENCES)
bg_wgs     = gpd.read_file(INPUT_GPKG, layer=LAYER_BACKGROUND)

_assert_crs_wgs84(dengue_wgs, "presences")
_assert_crs_wgs84(bg_wgs,     "background")

print(f"Presences:  {len(dengue_wgs)}")
print(f"Background: {len(bg_wgs)}")

# ── Cluster ───────────────────────────────────────────────
pres_labels, bg_labels, km = fit_presence_kmeans(dengue_wgs, bg_wgs, N_CLUSTERS, SEED)
cluster_stats = _summarize_clusters(pres_labels, bg_labels)

# ── Select test clusters ─────────────────────────────────
test_clusters = set(map(int, choose_test_clusters(
    cluster_stats, N_TEST_CLUSTERS, rng, strategy=TEST_SELECTION_STRATEGY
)))
print(f"\nTest clusters: {sorted(test_clusters)}")

# ── Assign CV folds ──────────────────────────────────────
cv_stats = cluster_stats[~cluster_stats["cluster"].isin(test_clusters)].copy()
fold_map = balanced_fold_assignment(cv_stats, N_FOLDS, rng)

# ── Build split GeoDataFrames ────────────────────────────
def _attach(gdf, labels):
    df = gdf.copy()
    df["cluster_id"] = labels
    df["split"] = np.where(df["cluster_id"].isin(test_clusters), "test", "cv")
    df["fold"]  = df["cluster_id"].map(fold_map).where(df["split"] == "cv", other=np.nan)
    df["in_test_buffer"] = False
    return df

pres_split = _attach(dengue_wgs, pres_labels)
bg_split   = _attach(bg_wgs,     bg_labels)

# ── Buffer flags ─────────────────────────────────────────
pres_split["in_test_buffer"] = compute_test_buffer_mask(pres_split, "split", TEST_BUFFER_KM)
bg_split["in_test_buffer"]   = compute_test_buffer_mask(bg_split,   "split", TEST_BUFFER_KM)

# ── Summary ──────────────────────────────────────────────
n_tp = (pres_split["split"] == "test").sum()
n_tb = (bg_split["split"] == "test").sum()
print(f"Presences  → test: {n_tp}, CV: {len(pres_split) - n_tp}")
print(f"Background → test: {n_tb}, CV: {len(bg_split) - n_tb}")
print(f"Buffer (≤{TEST_BUFFER_KM:.0f} km) → pres: {pres_split['in_test_buffer'].sum()}, "
      f"bg: {bg_split['in_test_buffer'].sum()}")

# fold balance table
sanity = (
    pres_split.query("split == 'cv'").groupby("fold").size().rename("n_pres")
    .to_frame()
    .join(bg_split.query("split == 'cv'").groupby("fold").size().rename("n_bg"), how="outer")
    .fillna(0).astype(int).sort_index()
)
print("\nCV fold balance:")
print(sanity)

## 12. Diagnostics & Plots

In [ ]:
if RUN_DIAGNOSTICS:
    leakage_diagnostics(pres_split)
    fold_separation_diagnostics(pres_split)

if RUN_PLOTS:
    plot_test_and_buffer(pres_split)
    plot_fold_counts(pres_split, bg_split)
    plot_cv_to_test_distances(pres_split)

## 13. Save Outputs

In [ ]:
# ── GeoPackage (full geometry + metadata) ─────────────────
pres_split.to_file(OUT_GPKG, layer=OUT_LAYER_PRES_SPLITS, driver="GPKG")
bg_split.to_file(OUT_GPKG,   layer=OUT_LAYER_BG_SPLITS,   driver="GPKG")
print(f"Saved GPKG layers → {OUT_GPKG}")

# ── CSV (for downstream model training) ──────────────────
export_test_cv_csv(pres_split, bg_split, TEST_CSV, CV_CSV)

## 14. (Optional) Reload and Re-export

In [ ]:
# If you need to re-export CSVs from an existing GeoPackage:
#
# pres_split = gpd.read_file(OUT_GPKG, layer=OUT_LAYER_PRES_SPLITS)
# bg_split   = gpd.read_file(OUT_GPKG, layer=OUT_LAYER_BG_SPLITS)
# export_test_cv_csv(pres_split, bg_split, TEST_CSV, CV_CSV)